# Personalized Medicine Recommendation

## Objective

The objective of this notebook is to develop a personalized
medicine recommendation approach using patient information
and disease-related data.

The analysis will consider patient characteristics and risk-related
information to support personalized recommendations.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/Cleaned_Dataset.csv")

df.head()

,disease,fever,cough,fatigue,difficulty_breathing,age,gender,blood_pressure,cholesterol_level,outcome_variable,age_scaled,bp_scaled,chol_scaled,risk_level
0,Influenza,Yes,No,Yes,Yes,19,female,0,1,Positive,-2.091160,-2.419529,-0.576777,Low
1,Common Cold,No,Yes,Yes,No,25,female,1,1,Negative,-1.631964,-0.723915,-0.576777,Low
2,Eczema,No,Yes,Yes,No,25,female,1,1,Negative,-1.631964,-0.723915,-0.576777,Medium
3,Asthma,Yes,Yes,No,Yes,25,male,1,1,Positive,-1.631964,-0.723915,-0.576777,Medium
4,Asthma,Yes,Yes,No,Yes,25,male,1,1,Positive,-1.631964,-0.723915,-0.576777,Medium


In [2]:
print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (349, 14)

Columns:
['disease', 'fever', 'cough', 'fatigue', 'difficulty_breathing', 'age', 'gender', 'blood_pressure', 'cholesterol_level', 'outcome_variable', 'age_scaled', 'bp_scaled', 'chol_scaled', 'risk_level']


In [3]:
df.isnull().sum()

disease                 0
fever                   0
cough                   0
fatigue                 0
difficulty_breathing    0
age                     0
gender                  0
blood_pressure          0
cholesterol_level       0
outcome_variable        0
age_scaled              0
bp_scaled               0
chol_scaled             0
risk_level              0
dtype: int64

In [4]:
df.duplicated().sum()

np.int64(49)

In [5]:
print("Unique diseases:", df["disease"].nunique())
print("\nDisease counts:")
print(df["disease"].value_counts())

Unique diseases: 116

Disease counts:
disease
Asthma                            23
Stroke                            16
Osteoporosis                      14
Hypertension                      10
Diabetes                          10
                                  ..
Autism Spectrum Disorder (ASD)     1
Hypoglycemia                       1
Fibromyalgia                       1
Eating Disorders (Anorexia         1
Williams Syndrome                  1
Name: count, Length: 116, dtype: int64


In [6]:
df["risk_level"].value_counts()

risk_level
Medium    198
High       90
Low        61
Name: count, dtype: int64

In [7]:
df["outcome_variable"].value_counts()

outcome_variable
Positive    186
Negative    163
Name: count, dtype: int64

In [8]:
df["age"].describe()

count    349.000000
mean      46.323782
std       13.085090
min       19.000000
25%       35.000000
50%       45.000000
75%       55.000000
max       90.000000
Name: age, dtype: float64

In [9]:
print("Gender:")
print(df["gender"].value_counts())

print("\nBlood Pressure:")
print(df["blood_pressure"].value_counts())

print("\nCholesterol Level:")
print(df["cholesterol_level"].value_counts())

Gender:
gender
female    176
male      173
Name: count, dtype: int64

Blood Pressure:
blood_pressure
2    167
1    164
0     18
Name: count, dtype: int64

Cholesterol Level:
cholesterol_level
2    166
1    149
0     34
Name: count, dtype: int64


In [10]:
id_cols = df.columns.tolist()

duplicate_rows = df[df.duplicated(keep=False)]

print("Total duplicate-related rows:", len(duplicate_rows))
print("\nFirst duplicate group:")
print(duplicate_rows.head(10))

Total duplicate-related rows: 98

First duplicate group:
            disease fever cough fatigue difficulty_breathing  age  gender  \
3            Asthma   Yes   Yes      No                  Yes   25    male   
4            Asthma   Yes   Yes      No                  Yes   25    male   
6         Influenza   Yes   Yes     Yes                  Yes   25  female   
7         Influenza   Yes   Yes     Yes                  Yes   25  female   
8   Hyperthyroidism    No   Yes      No                   No   28  female   
9   Hyperthyroidism    No   Yes      No                   No   28  female   
22     Dengue Fever   Yes    No     Yes                   No   30  female   
23     Dengue Fever   Yes    No     Yes                   No   30  female   
34           Asthma   Yes   Yes      No                  Yes   30  female   
35           Asthma   Yes   Yes      No                  Yes   30  female   

    blood_pressure  cholesterol_level outcome_variable  age_scaled  bp_scaled  \
3             

## Prepare Data for Analysis

The dataset contains 49 exact duplicate records.
For further patient-level analysis, we will create a duplicate-free
copy while keeping the original dataset unchanged.

In [11]:
df_clean = df.drop_duplicates().copy()

print("Original dataset shape:", df.shape)
print("After removing duplicates:", df_clean.shape)
print("Duplicates remaining:", df_clean.duplicated().sum())

Original dataset shape: (349, 14)
After removing duplicates: (300, 14)
Duplicates remaining: 0


In [12]:
print("Age scaled:")
print(df_clean["age_scaled"].describe())

print("\nBP scaled:")
print(df_clean["bp_scaled"].describe())

print("\nCholesterol scaled:")
print(df_clean["chol_scaled"].describe())

Age scaled:
count    300.000000
mean      -0.043403
std        0.964046
min       -2.091160
25%       -0.866638
50%       -0.101312
75%        0.664013
max        3.342654
Name: age_scaled, dtype: float64

BP scaled:
count    300.000000
mean      -0.017410
std        1.023384
min       -2.419529
25%       -0.723915
50%       -0.723915
75%        0.971698
max        0.971698
Name: bp_scaled, dtype: float64

Cholesterol scaled:
count    300.000000
mean      -0.017624
std        1.004241
min       -2.101740
25%       -0.576777
50%       -0.576777
75%        0.948186
max        0.948186
Name: chol_scaled, dtype: float64


In [13]:
df_clean[[
    "age",
    "gender",
    "blood_pressure",
    "cholesterol_level",
    "outcome_variable",
    "risk_level"
]].head(10)

,age,gender,blood_pressure,cholesterol_level,outcome_variable,risk_level
0,19,female,0,1,Positive,Low
1,25,female,1,1,Negative,Low
2,25,female,1,1,Negative,Medium
3,25,male,1,1,Positive,Medium
5,25,female,1,1,Positive,Medium
6,25,female,1,1,Positive,Low
8,28,female,1,1,Negative,Medium
10,28,male,2,1,Positive,Medium
11,29,female,1,0,Negative,Low
12,29,female,1,2,Negative,Medium


In [14]:
df_clean[[
    "disease",
    "age",
    "gender",
    "blood_pressure",
    "cholesterol_level",
    "risk_level",
    "outcome_variable"
]].head(10)

,disease,age,gender,blood_pressure,cholesterol_level,risk_level,outcome_variable
0,Influenza,19,female,0,1,Low,Positive
1,Common Cold,25,female,1,1,Low,Negative
2,Eczema,25,female,1,1,Medium,Negative
3,Asthma,25,male,1,1,Medium,Positive
5,Eczema,25,female,1,1,Medium,Positive
6,Influenza,25,female,1,1,Low,Positive
8,Hyperthyroidism,28,female,1,1,Medium,Negative
10,Asthma,28,male,2,1,Medium,Positive
11,Allergic Rhinitis,29,female,1,0,Low,Negative
12,Anxiety Disorders,29,female,1,2,Medium,Negative


In [15]:
pd.crosstab(
    df_clean["disease"],
    df_clean["risk_level"]
).head(20)

risk_level,High,Low,Medium
disease,,,
Acne,0,1,0
Allergic Rhinitis,0,6,0
Alzheimer's Disease,3,0,0
Anemia,0,0,1
Anxiety Disorders,0,0,6
Appendicitis,0,1,0
Asthma,0,0,16
Atherosclerosis,0,0,1
Autism Spectrum Disorder (ASD),0,0,1


## Role of the Patient/Risk Dataset

`Cleaned_Dataset.csv` is used in this notebook for supporting
patient-profile and risk analysis.

It is not used as the diagnosis source for the disease prediction model.
The disease prediction model uses `Training.csv`, and medication
information is retrieved from `medications.csv`.


## Load Medicine Recommendation Dataset

The medications dataset contains disease-wise medication information.
We will first inspect its structure before building the recommendation system.

In [16]:
medications_df = pd.read_csv("../data/medications.csv")

print("Dataset shape:", medications_df.shape)
print("\nColumns:")
print(medications_df.columns.tolist())

medications_df.head()

Dataset shape: (41, 2)

Columns:
['Disease', 'Medication']


,Disease,Medication
0,Fungal infection,"['Antifungal Cream', 'Fluconazole', 'Terbinafi..."
1,Allergy,"['Antihistamines', 'Decongestants', 'Epinephri..."
2,GERD,"['Proton Pump Inhibitors (PPIs)', 'H2 Blockers..."
3,Chronic cholestasis,"['Ursodeoxycholic acid', 'Cholestyramine', 'Me..."
4,Drug Reaction,"['Antihistamines', 'Epinephrine', 'Corticoster..."


In [17]:
training_df = pd.read_csv("../data/Training.csv")

print("Training dataset shape:", training_df.shape)
print("Target column:", training_df.columns[-1])
print("Number of diseases:", training_df["prognosis"].nunique())

Training dataset shape: (4920, 133)
Target column: prognosis
Number of diseases: 41


In [18]:
training_diseases = set(training_df["prognosis"].unique())
medicine_diseases = set(medications_df["Disease"].unique())

print("Diseases in Training.csv:", len(training_diseases))
print("Diseases in medications.csv:", len(medicine_diseases))

print("\nDiseases missing in medications.csv:")
print(training_diseases - medicine_diseases)

print("\nDiseases missing in Training.csv:")
print(medicine_diseases - training_diseases)

Diseases in Training.csv: 41
Diseases in medications.csv: 41

Diseases missing in medications.csv:
{'Diabetes ', '(vertigo) Paroymsal  Positional Vertigo', 'Hypertension ', 'Peptic ulcer diseae'}

Diseases missing in Training.csv:
{'Peptic ulcer disease', 'Diabetes', '(vertigo) Paroymsal Positional Vertigo', 'Hypertension'}


In [19]:
training_diseases_clean = {
    disease.strip() for disease in training_df["prognosis"]
}

medicine_diseases_clean = {
    disease.strip() for disease in medications_df["Disease"]
}

print("Training diseases after removing extra spaces:", len(training_diseases_clean))
print("Medicine diseases after removing extra spaces:", len(medicine_diseases_clean))

print("\nStill unmatched:")
print(training_diseases_clean - medicine_diseases_clean)

Training diseases after removing extra spaces: 41
Medicine diseases after removing extra spaces: 41

Still unmatched:
{'(vertigo) Paroymsal  Positional Vertigo', 'Peptic ulcer diseae'}


In [20]:
disease_mapping = {
    "Peptic ulcer diseae": "Peptic ulcer disease",
    "(vertigo) Paroymsal  Positional Vertigo":
        "(vertigo) Paroymsal Positional Vertigo"
}

for training_name, medicine_name in disease_mapping.items():
    print(f"Training:    {training_name}")
    print(f"Medication:  {medicine_name}")
    print(f"Medicine entry exists:",
          medicine_name in medicine_diseases_clean)
    print()

Training:    Peptic ulcer diseae
Medication:  Peptic ulcer disease
Medicine entry exists: True

Training:    (vertigo) Paroymsal  Positional Vertigo
Medication:  (vertigo) Paroymsal Positional Vertigo
Medicine entry exists: True



## Disease-Medication Mapping

The disease prediction model and medication dataset may contain
minor differences in disease names.

A normalized disease name will be used to connect the predicted
disease with its corresponding medication information.

In [21]:
# Create a working copy of the medication dataset
medication_map = medications_df.copy()

# Remove leading and trailing spaces from disease names
medication_map["Disease_clean"] = medication_map["Disease"].str.strip()

# Verified disease-name mappings
disease_mapping = {
    "Peptic ulcer diseae": "Peptic ulcer disease",
    "(vertigo) Paroymsal  Positional Vertigo":
        "(vertigo) Paroymsal Positional Vertigo"
}

# Convert medication dataset names to the names
# used by the disease prediction dataset
reverse_mapping = {
    medicine_name: training_name
    for training_name, medicine_name in disease_mapping.items()
}

medication_map["Disease_clean"] = medication_map["Disease_clean"].replace(
    reverse_mapping
)

# Verify the final mapping
print("Medication mapping shape:", medication_map.shape)

print("\nSample records:")
print(
    medication_map[
        ["Disease", "Disease_clean", "Medication"]
    ].head()
)

Medication mapping shape: (41, 3)

Sample records:
               Disease        Disease_clean  \
0     Fungal infection     Fungal infection   
1              Allergy              Allergy   
2                 GERD                 GERD   
3  Chronic cholestasis  Chronic cholestasis   
4        Drug Reaction        Drug Reaction   

                                          Medication  
0  ['Antifungal Cream', 'Fluconazole', 'Terbinafi...  
1  ['Antihistamines', 'Decongestants', 'Epinephri...  
2  ['Proton Pump Inhibitors (PPIs)', 'H2 Blockers...  
3  ['Ursodeoxycholic acid', 'Cholestyramine', 'Me...  
4  ['Antihistamines', 'Epinephrine', 'Corticoster...  


In [22]:
# Verify complete disease coverage

training_disease_list = sorted(
    training_df["prognosis"].str.strip().unique()
)

medication_disease_list = sorted(
    medication_map["Disease_clean"].unique()
)

missing_medications = sorted(
    set(training_disease_list) - set(medication_disease_list)
)

print("Total diseases in Training.csv:", len(training_disease_list))
print("Total diseases in medications.csv:", len(medication_disease_list))

print("\nDiseases without medication information:")
print(missing_medications)

print("\nNumber of diseases without medication information:",
      len(missing_medications))

Total diseases in Training.csv: 41
Total diseases in medications.csv: 41

Diseases without medication information:
[]

Number of diseases without medication information: 0


## Load the Trained Disease Prediction Model

The trained disease prediction model and disease encoder
created in the disease prediction notebook are loaded here.

The model will be used to predict the disease before
retrieving the corresponding medication information.

In [23]:
import joblib

# Load the trained disease prediction model
svc_model = joblib.load("../models/best_model.pkl")

# Load the disease label encoder
disease_encoder = joblib.load("../models/disease_encoder.pkl")

print("Disease prediction model loaded successfully.")
print("Number of disease classes:", len(disease_encoder.classes_))

Disease prediction model loaded successfully.
Number of disease classes: 41


## Disease Prediction and Medication Recommendation

The trained disease prediction model predicts a disease from the
symptom features in `Training.csv`.

The predicted disease is then matched with `medications.csv` to
retrieve the corresponding medication information.


In [24]:
# Select one patient record from Training.csv
sample_index = 0

sample_record = training_df.iloc[sample_index]

# Symptoms for disease prediction
sample_symptoms = training_df.drop(columns=["prognosis"]).iloc[[sample_index]]

# Actual disease from the same record
actual_disease = sample_record["prognosis"]

# Predict disease
predicted_encoded = svc_model.predict(sample_symptoms)[0]
predicted_disease = disease_encoder.inverse_transform(
    [predicted_encoded]
)[0]

print("Actual Disease:   ", actual_disease)
print("Predicted Disease:", predicted_disease)

Actual Disease:    Fungal infection
Predicted Disease: Fungal infection


## Recommendation Function

The predicted disease is used as the diagnosis for medication lookup.

Patient profile information can be supplied separately by the final
application as contextual information. It is not used to independently
rank or select medications because the available medication dataset
does not contain a patient-level medication-selection target.


In [25]:
def generate_recommendation(predicted_disease, patient_profile):
    
    # Find medication information using the predicted disease
    result = medication_map[
        medication_map["Disease_clean"] == predicted_disease
    ]

    if result.empty:
        return {
            "status": "No medication information found",
            "predicted_disease": predicted_disease,
            "patient_profile": patient_profile,
            "medications": []
        }

    medications = result["Medication"].iloc[0]

    return {
        "status": "Recommendation generated",
        "predicted_disease": predicted_disease,
        "patient_profile": patient_profile,
        "medications": medications
    }

In [26]:
# Generate a recommendation for the predicted disease

patient_context = {
    "record_index": sample_index,
    "actual_disease": actual_disease
}

final_recommendation = generate_recommendation(
    predicted_disease=predicted_disease,
    patient_profile=patient_context
)

print("Status:", final_recommendation["status"])
print("Predicted Disease:", final_recommendation["predicted_disease"])
print("Medication Information:", final_recommendation["medications"])


Status: Recommendation generated
Predicted Disease: Fungal infection
Medication Information: ['Antifungal Cream', 'Fluconazole', 'Terbinafine', 'Clotrimazole', 'Ketoconazole']


## Conclusion and Limitations

The notebook demonstrates the core recommendation pipeline:

1. Predict a disease from symptom features using the trained model.
2. Match the predicted disease with the medication dataset.
3. Return the available medication information with patient context.

Limitations:

- The medication dataset provides disease-wise medication information,
  not patient-level treatment outcomes.
- Therefore, the system retrieves available medication information rather
  than learning which medication is medically optimal for an individual.
- Patient profile attributes are retained as contextual information and
  are not used to make unsupported medication-selection claims.
